# Week5_ex1 - Force Between Two Magnets

Two permanent magnets stacked on the z-axis with a small gap between them, using virtual force to see how strong the pull actually is.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil


In [ ]:
# 1. Launch AEDT
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. Turn off autosave
DT.disable_autosave()

# 3. Create project and set solution type
sol_type = "Magnetostatic"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. odesign object for script-recording style calls
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

proj_name = "Week5_ex1"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

desi_name = "Week5_ex1"

In [ ]:
## Save project and apply design name ##

proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()

In [ ]:
# Magnet height, radius, gap between the two magnets, and segment count for the polygon cross-section
m_h = 15
M3D["m_h"] = f"{m_h}mm"

m_r = 10
M3D["m_r"] = f"{m_r}mm"

gap = 5
M3D["gap"] = f"{gap}mm"

line_num_seg = 18
M3D["line_num_seg"] = f"{line_num_seg}"

In [ ]:
# Duplicate NdFe35 twice so each magnet can be magnetized in a different direction
magnet_n = M3D.materials.duplicate_material(material="NdFe35", name="magnet_n")
magnet_n.get_magnetic_coercivity()  # check current magnetization
magnet_n.set_magnetic_coercivity(value=-890000, x=0, y=0, z=1)  # magnetized along +z

magnet_s = M3D.materials.duplicate_material(material="NdFe35", name="magnet_s")
magnet_s.get_magnetic_coercivity()  # check current magnetization
magnet_s.set_magnetic_coercivity(value=-890000, x=0, y=0, z=-1)  # magnetized along -z

In [ ]:
# Magnetostatic setup, nonlinear BH curve mode on
my_setup = M3D.create_setup(name="Setup1")
my_setup.props["MaximumPasses"] = 20
my_setup.props["MinimumPasses"] = 2
my_setup.props["MinimumConvergedPasses"] = 1
my_setup.props["PercentRefinement"] = 15
my_setup.props["PercentError"] = 1
my_setup.props["MuOption"] = {"MuNonLinearBH": True}
my_setup.update()

In [ ]:
# Two magnets stacked along z with a gap between them
origin = ["m_r", 0, 0]
magnet_1 = M3D.modeler.create_polyhedron(orientation="Z", center=[0.0, 0.0, 0.0], origin=origin,
                                        height="m_h", num_sides=line_num_seg, name="magnet_1", material=magnet_n)

origin = ["m_r", 0, "m_h+gap"]
magnet_2 = M3D.modeler.create_polyhedron(orientation="Z", center=[0.0, 0.0, "m_h+gap"], origin=origin,
                                         height="m_h", num_sides=line_num_seg, name="magnet_2", material=magnet_n)

# Dummy object just so the mesh in the gap region can be controlled separately
origin = ["-m_r*20", 0, "-m_h*10"]
dummy = M3D.modeler.create_polyhedron(orientation="Z", center=[0.0, 0.0, "-m_h*10"], origin=origin, height="m_h*20+m_h*2+gap",
                                      num_sides=line_num_seg, name="dummy", material=None)

# Cross-section face through the dummy, used later for the field plot
M3D.modeler.section(assignment=dummy, plane="ZX", create_new=True, section_cross_object=False)
field_face = M3D.modeler.sheet_objects[-1]

# Background region
region = M3D.modeler.create_region(pad_value=300, pad_type='Percentage Offset', name='Region')

In [ ]:
# NOTE: MaxLength coarsened (m_h*20/5 -> m_h*20/2 for the dummy region,
# m_r/5 -> m_r/2 for the magnets), and PercentRefinement dropped from 30 to 15 --
# at the original settings this pushed the mesh to ~74,614 elements, over the
# Student license's cap. Coarser mesh + slower per-pass refinement keeps it
# under the cap while still smoothing the field plot.

oModule = oDesign.GetModule("MeshSetup")

oModule.AssignLengthOp(
	[
		"NAME:Length1",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["dummy"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "m_h*20/2"
	])

oModule.AssignLengthOp(
	[
		"NAME:Length2",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["magnet_1","magnet_2"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "m_r/2"
	])

In [ ]:
# Virtual force on each magnet so we can read off the attraction/repulsion after solving
M3D.assign_force(assignment=magnet_1, coordinate_system='Global', is_virtual=True, force_name="force_1")

M3D.assign_force(assignment=magnet_2, coordinate_system='Global', is_virtual=True, force_name="force_2")

In [ ]:
my_setup.analyze()

In [ ]:
oModule = oDesign.GetModule("ReportSetup")

oModule.CreateReport("Force Table 1", "Magnetostatic", "Data Table", "Setup1 : LastAdaptive", [],
	[
		"m_h:="			, ["All"],
		"m_r:="			, ["Nominal"],
		"gap:="			, ["Nominal"],
		"line_num_seg:="	, ["Nominal"]
	],
	[
		"Y Component:="		, ["force_1.Force_mag","force_1.Force_x","force_1.Force_y","force_1.Force_z","force_2.Force_mag","force_2.Force_x","force_2.Force_y","force_2.Force_z"]
	])

In [ ]:
## Field plot ##

oModule = oDesign.GetModule("FieldsReporter")
oModule.CreateFieldPlot(
	[
		"NAME:B_Vector1",
		"SolutionName:="	, "Setup1 : LastAdaptive",
		"UserSpecifyName:="	, 0,
		"UserSpecifyFolder:="	, 0,
		"QuantityName:="	, "B_Vector",
		"PlotFolder:="		, "B",
		"StreamlinePlot:="	, False,
		"AdjacentSidePlot:="	, False,
		"FullModelPlot:="	, False,
		"IntrinsicVar:="	, "",
		"PlotGeomInfo:="	, [1,"Surface","FacesList",1,f"{field_face.id+1}"],
		"FilterBoxes:="		, [0],
		[
			"NAME:PlotOnSurfaceSettings",
			"Filled:="		, False,
			"IsoValType:="		, "Tone",
			"AddGrid:="		, False,
			"MapTransparency:="	, True,
			"Refinement:="		, 0,
			"Transparency:="	, 0,
			"SmoothingLevel:="	, 0,
			"ShadingType:="		, 0,
			[
				"NAME:Arrow3DSpacingSettings",
				"ArrowUniform:="	, True,
				"ArrowSpacing:="	, 0,
				"MinArrowSpacing:="	, 0,
				"MaxArrowSpacing:="	, 0
			],
			"GridColor:="		, [255,255,255]
		],
		"EnableGaussianSmoothing:=", False,
		"SurfaceOnly:="		, False
	], "Field")

oModule.SetPlotFolderSettings("B",
	[
		"NAME:FieldsPlotSettings",
		"Real time mode:="	, True,
		[
			"NAME:ColorMapSettings",
			"ColorMapType:="	, "Spectrum",
			"SpectrumType:="	, "Rainbow",
			"UniformColor:="	, [127,255,255],
			"RampColor:="		, [255,127,127]
		],
		[
			"NAME:Scale3DSettings",
			"unit:="		, 165,
			"m_nLevels:="		, 10,
			"minvalue:="		, 6.9E-05,
			"maxvalue:="		, 0.05,
			"log:="			, False,
			"IntrinsicMin:="	, 6.90271485267979E-05,
			"IntrinsicMax:="	, 1.18608944173731,
			"LimitFieldValuePrecision:=", False,
			"FieldValuePrecisionDigits:=", 4,
			"dB:="			, False,
			"AnimationStaticScale:=", False,
			"ScaleType:="		, 1,
			"UserSpecifyValues:="	, [11,6.90271481289528E-05,0.11867106705904,0.237273097038269,0.355875134468079,0.474477171897888,0.593079209327698,0.711681246757507,0.830283284187317,0.948885321617126,1.06748735904694,1.18608939647675],
			"ValueNumberFormatTypeAuto:=", 0,
			"ValueNumberFormatTypeScientific:=", False,
			"ValueNumberFormatWidth:=", 7,
			"ValueNumberFormatPrecision:=", 3
		],
		[
			"NAME:Marker3DSettings",
			"MarkerType:="		, 9,
			"MarkerMapSize:="	, False,
			"MarkerMapColor:="	, False,
			"MarkerSize:="		, 0.25
		],
		[
			"NAME:Arrow3DSettings",
			"ArrowType:="		, 1,
			"ArrowMapSize:="	, False,
			"ArrowMapColor:="	, True,
			"ShowArrowTail:="	, False,
			"ArrowSize:="		, 0.0495999976992607,
			"ArrowMinMagnitude:="	, -0.499931,
			"ArrowMaxMagnitude:="	, 1.68609,
			"ArrowMagnitudeThreshold:=", 0,
			"ArrowMagnitudeFilteringFlag:=", False,
			"ArrowMinIntrinsicMagnitude:=", 0,
			"ArrowMaxIntrinsicMagnitude:=", 1
		],
		[
			"NAME:DeformScaleSettings",
			"ShowDeformation:="	, True,
			"MinScaleFactor:="	, 0,
			"MaxScaleFactor:="	, 1,
			"DeformationScale:="	, 0,
			"ShowDeformationOutline:=", False
		]
	])

oModule.SetFieldPlotSettings("B_Vector1",
	[
		"NAME:FieldsPlotItemSettings",
		[
			"NAME:PlotOnSurfaceSettings",
			"Filled:="		, False,
			"IsoValType:="		, "Tone",
			"AddGrid:="		, False,
			"MapTransparency:="	, True,
			"Refinement:="		, 0,
			"Transparency:="	, 0,
			"SmoothingLevel:="	, 0,
			"ShadingType:="		, 0,
			[
				"NAME:Arrow3DSpacingSettings",
				"ArrowUniform:="	, True,
				"ArrowSpacing:="	, 3.17870092391968,
				"MinArrowSpacing:="	, 3,
				"MaxArrowSpacing:="	, 20.8701000213623
			],
			"GridColor:="		, [255,255,255]
		]
	])

In [ ]:
## Save project ##

M3D.save_project()